# Module P08 — Console J2 et boucle de jeu

**Objectif :** coder `phase_j2()` et `boucle_de_jeu()` — l'orchestrateur complet.

> Compatible Google Colab — aucun fichier externe requis.

## Section 1 — Rappels synthétiques

### Ce qu'on a vu dans P07 (prérequis)
- `phase_j1()` : boucle saisie → parser → valider → exécuter (max 3 déplacements)
- `position_depuis_chaine()` : convertit 'B4' en (col=1, lig=3)

### Structure d'un tour complet
```
while not etat['partie_finie']:
    1. phase_j1(etat)          # J1 joue (drones)
    2. phase_j2(etat)          # J2 joue (tempêtes)
    3. deplacer_tempetes(etat) # météo automatique
    4. verifier_fin_partie()   # fin ?
    5. etat['tour'] += 1
```

### Distance de Chebyshev (validation J2)
Pour une tempête en (col=5, lig=5), les cases accessibles en 1 déplacement :
```
max(|col_cible - 5|, |lig_cible - 5|) <= 1
```
Cela inclut les 8 cases autour (haut, bas, gauche, droite, diagonales).

In [ ]:
import random
random.seed(99)
COLS = 'ABCDEFGHIJKL'
MAX_DEPLACEMENTS_J2 = 2
GRILLE_TAILLE = 12

def position_depuis_chaine(chaine):
    try:
        col = COLS.index(chaine[0].upper())
        lig = int(chaine[1:]) - 1
        if 0 <= col < 12 and 0 <= lig < 12:
            return col, lig
    except (ValueError, IndexError):
        pass
    return None, None

def deplacer_tempetes(etat):
    for tid, t in etat['tempetes'].items():
        if random.random() < 0.5:
            etat['grille'][t['lig']][t['col']] = '.'
            t['col'] = max(0, min(11, t['col'] + t['dx']))
            t['lig'] = max(0, min(11, t['lig'] + t['dy']))
            etat['grille'][t['lig']][t['col']] = tid

def verifier_fin_partie(etat, TOURS_MAX=5):
    if all(s['etat'] == 'sauve' for s in etat['survivants'].values()):
        etat['partie_finie'] = True; etat['victoire'] = True
        return 'Victoire !'
    if all(d['hors_service'] for d in etat['drones'].values()):
        etat['partie_finie'] = True; etat['victoire'] = False
        return 'Défaite — drones'
    if etat['tour'] >= TOURS_MAX:
        etat['partie_finie'] = True; etat['victoire'] = False
        return f'Défaite — {TOURS_MAX} tours'
    return ''

def creer_etat_test():
    grille = [['.' for _ in range(12)] for _ in range(12)]
    etat = {
        'tour': 1, 'score': 0, 'partie_finie': False, 'victoire': False,
        'grille': grille, 'hopital': (0, 0), 'batiments': [],
        'drones': {
            'D1': {'id': 'D1', 'col': 3, 'lig': 3, 'batterie': 15,
                   'batterie_max': 20, 'survivant': None,
                   'bloque': 0, 'hors_service': False}
        },
        'tempetes': {
            'T1': {'id': 'T1', 'col': 5, 'lig': 5, 'dx': 1, 'dy': 0}
        },
        'survivants': {
            'S1': {'id': 'S1', 'col': 4, 'lig': 4, 'etat': 'en_attente'}
        },
        'zones_x': set(), 'historique': []
    }
    etat['grille'][0][0] = 'H'
    etat['grille'][3][3] = 'D1'
    etat['grille'][5][5] = 'T1'
    etat['grille'][4][4] = 'S1'
    return etat

print('Stubs chargés ✓')

## Section 2 — Exercices guidés

### Étape 1 — Validation mouvement tempête (distance Chebyshev)

In [ ]:
def valider_mouvement_tempete(etat, id_tempete, col_cible, lig_cible):
    """Valide un déplacement de tempête : distance ≤1, dans la grille."""    t = etat['tempetes'].get(id_tempete)
    if t is None:
        return False, f'{id_tempete} inconnue'

    # TODO 1 : vérifier que col_cible et lig_cible sont dans [0, 11]
    # if not (0 <= col_cible < 12 and 0 <= lig_cible < 12):
    #     return False, 'hors grille'

    # TODO 2 : calculer la distance de Chebyshev
    # dist = max(abs(col_cible - t['col']), abs(lig_cible - t['lig']))
    # if dist > 1:
    #     return False, f'distance {dist} > 1 case'

    return True, 'ok'

In [ ]:
# Tests validation tempête — T1 est en (col=5, lig=5)
etat = creer_etat_test()

ok, _ = valider_mouvement_tempete(etat, 'T1', 6, 5)  # +1 col — OK
assert ok, 'Déplacement adjacent doit être valide'

ok2, msg2 = valider_mouvement_tempete(etat, 'T1', 8, 5)  # +3 col — trop loin
assert not ok2, f'Distance 3 doit être refusée : {msg2}'

ok3, msg3 = valider_mouvement_tempete(etat, 'T1', -1, 5)  # hors grille
assert not ok3, 'Hors grille doit être refusé'

print('✓ Étape 1 validée — validation tempête OK')

### Étape 2 — Phase J2 avec saisie simulée

In [ ]:
def phase_j2(etat, input_fn=input):
    """Phase J2 : jusqu'à MAX_DEPLACEMENTS_J2 déplacements de tempêtes."""    nb_depl = 0
    logs = []

    while nb_depl < MAX_DEPLACEMENTS_J2:
        saisie = input_fn(f'J2 [{nb_depl}/{MAX_DEPLACEMENTS_J2}] > ').strip()

        if saisie.lower() in ('fin', 'passe', ''):
            break

        parties = saisie.split()
        if len(parties) != 2:
            print('Format : T1 F6  |  fin pour terminer')
            continue

        id_tempete, pos_str = parties[0].upper(), parties[1]

        # TODO 1 : vérifier que id_tempete est dans etat['tempetes']
        # if id_tempete not in etat['tempetes']:
        #     print(...)
        #     continue

        col, lig = position_depuis_chaine(pos_str)
        if col is None:
            print(f"Position '{pos_str}' invalide")
            continue

        ok, msg = valider_mouvement_tempete(etat, id_tempete, col, lig)
        if not ok:
            print(f'Refusé : {msg}')
            continue

        # TODO 2 : exécuter le déplacement de la tempête
        # t = etat['tempetes'][id_tempete]
        # etat['grille'][t['lig']][t['col']] = '.'
        # t['col'], t['lig'] = col, lig
        # etat['grille'][lig][col] = id_tempete
        # log = f'{id_tempete} → ({col},{lig})'
        # logs.append(log)
        # nb_depl += 1

    return logs

In [ ]:
# Test : J2 déplace T1 de (5,5) vers (6,5)
etat = creer_etat_test()
saisies = iter(['T1 G6', 'fin'])  # G=col6, 6=lig5
logs = phase_j2(etat, input_fn=lambda p='': next(saisies))

print('Logs J2 :', logs)
assert etat['tempetes']['T1']['col'] == 6, f"T1 col attendu 6, obtenu {etat['tempetes']['T1']['col']}"
assert etat['tempetes']['T1']['lig'] == 5, f"T1 lig attendu 5, obtenu {etat['tempetes']['T1']['lig']}"
print('✓ Étape 2 validée — phase J2 OK')

### Étape 3 — Boucle de jeu complète
La `boucle_de_jeu()` orchestre les 4 phases. Elle s'arrête quand `etat['partie_finie']` est `True`.

In [ ]:
def boucle_de_jeu(etat, phase_j1_fn, phase_j2_fn, TOURS_MAX=5):
    """Orchestre le jeu : J1 → J2 → météo → fin, tour après tour."""    while not etat['partie_finie']:
        print(f"--- Tour {etat['tour']} | Score : {etat['score']} ---")

        # 1. Phase J1
        phase_j1_fn(etat)
        if etat['partie_finie']:
            break

        # TODO 1 : Phase J2
        # phase_j2_fn(etat)
        # if etat['partie_finie']:
        #     break

        # TODO 2 : Phase météo automatique
        # deplacer_tempetes(etat)

        # TODO 3 : Vérifier fin de partie AVANT d'incrémenter le tour
        # msg_fin = verifier_fin_partie(etat, TOURS_MAX)
        # etat['tour'] += 1
        # if msg_fin:
        #     print(msg_fin)
        #     break
        etat['tour'] += 1  # à supprimer quand TODO 3 est fait

In [ ]:
# Test boucle complète — fin par limite de tours (TOURS_MAX=3)
etat = creer_etat_test()

# J1 et J2 passent toujours (saisies simulées avec 'fin')
def j1_simule(e): pass  # J1 ne joue rien
def j2_simule(e): pass  # J2 ne joue rien

boucle_de_jeu(etat, j1_simule, j2_simule, TOURS_MAX=3)

assert etat['partie_finie'] == True, 'La partie doit être terminée'
assert etat['victoire'] == False, 'Défaite attendue (tours max)'
print(f'Partie terminée au tour {etat["tour"]}, victoire={etat["victoire"]}')print('\n✓ Module P08 complet !')